# RNAbpFlow — Pilot10 predicted-IPKnot single-map baseline (Colab T4)

**Research:** RNA 3D structure prediction, frozen pilot10; **model:** RNAbpFlow (NOT DRfold). **Input:** RNA FASTA plus **predicted** IPKnot base-pair map from the existing NuFold pilot frozen inputs, not native/experimental structures or native base pairs. Official RNAbpFlow permits one `L×L` map, replicated internally into three channels. This run is a **predicted-BP-conditioned** condition, not MSA or sequence-only.

- DRfold/RNAbpFlow checkpoints are different; use the RNAbpFlow-specific `RNA3DB.ckpt`.
- Default: **one target `9g4p_B`, one sample**, fixed seed per upstream code; no cherry-picking. After successful smoke, set `SMOKE_ONLY=False` to run remaining targets with automatic skip and Drive persistence.
- Prediction outputs, metadata, and frozen input BP maps save to `MyDrive/RNA3D_Benchmark/RNAbpFlow/runs/rnabpflow_pilot10_ipknot_singlemap_v1/` after each target.
- **Colab T4 GPU** needed for inference; official environment was previously verified in Colab with Python 3.10, PyTorch 2.1.2 and NumPy 1.26.4. A new runtime must reinstall dependencies. Do not use native coordinates during inference.
- **Scientific limitation:** IPKnot can predict pseudoknotted pairs, and its predictions are not a substitute for true homolog-derived MSA. RNAbpFlow's own paper compares multiple base-pair predictors. This is a deliberately labelled *single predicted-BP-map* condition.


## 1 · Mount the correct Google Drive and choose scope

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, time, threading, shutil, json, hashlib, csv, re, zipfile, tempfile
from datetime import datetime, timezone
import numpy as np

PROJECT_ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
FROZEN_INPUTS=PROJECT_ROOT/'NuFold/runs/pilot10_single_20261018T203832Z/inputs'
RUN_ID='rnabpflow_pilot10_ipknot_singlemap_v1'
RUN_ROOT=PROJECT_ROOT/'RNAbpFlow/runs'/RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
assert FROZEN_INPUTS.is_dir(), f'Wrong Drive account or missing frozen inputs: {FROZEN_INPUTS}'

TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
EXPECTED_LENGTHS={'8ubd_H':36,'9fia_bJ':71,'9cfn_B':59,'9m31_B':51,'9g4p_B':33,'9lq7_K':49,'8z8u_B':41,'9g6k_lB':45,'9cf2_W':61,'8wf8_B':66}
SMOKE_ONLY=True                 # change to False after a successful 9g4p_B smoke
TARGETS_PER_SESSION=2          # applies when SMOKE_ONLY=False; repeat notebook to continue
SAMPLES_PER_TARGET=1           # frozen selection: Sample_0, no native-guided best-of-N
UPSTREAM_SHA='e8b1c077f7951d4d6ebe00c851d22c37777dfc9f'

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()

print('Output Drive folder:', RUN_ROOT)
print('Host Python:', sys.version.split()[0])
subprocess.run(['nvidia-smi'],check=True)


## 2 · Read frozen FASTA and predicted IPKnot dot-bracket maps, preflight all 10 targets

In [ ]:
PAIRS={'(':')','[':']','{':'}','<':'>'}
CLOSE={v:k for k,v in PAIRS.items()}
# Separate stacks allow crossing pseudo-knot brackets, but each bracket family must balance.
def dbn_to_binary_bp(dbn):
    n=len(dbn); mat=np.zeros((n,n),dtype=np.float32)
    stacks={k:[] for k in PAIRS}
    for i,ch in enumerate(dbn):
        if ch in PAIRS:stacks[ch].append(i)
        elif ch in CLOSE:
            opening=CLOSE[ch]
            assert stacks[opening],f'Unmatched closing bracket {ch} at {i}'
            j=stacks[opening].pop()
            mat[i,j]=mat[j,i]=1.0
        elif ch!='.':
            raise ValueError(f'Unsupported IPKnot dot-bracket symbol {ch!r}: do not silently discard predicted pairs')
    assert all(not v for v in stacks.values()),'Unclosed dot-bracket symbols'
    assert np.all(mat==mat.T) and np.all(np.diag(mat)==0)
    assert np.max(mat.sum(0))<=1, 'One nucleotide predicted in multiple base pairs: inspect IPKnot file'
    return mat

def get_inputs(target):
    p=FROZEN_INPUTS/target
    fa=p/f'{target}.fasta'; ss=p/f'{target}.single.ipknot.ss'
    assert fa.is_file() and ss.is_file(), f'Missing frozen FASTA or IPKnot prediction for {target}: {p}'
    def read_records(path):
        lines=[x.strip() for x in path.read_text().splitlines() if x.strip()]
        assert len(lines)>=2 and lines[0].startswith('>'),f'Unexpected input file format: {path}'
        assert not any(x.startswith('>') for x in lines[1:]),f'Multiple records in {path}'
        return lines[0][1:].split()[0],lines[1:]
    id1, seqrows=read_records(fa); id2, ssrows=read_records(ss)
    seq=''.join(seqrows).upper(); assert id1==target and id2==target
    assert set(seq)<=set('ACGU'),f'Noncanonical target sequence {target}'
    assert len(seq)==EXPECTED_LENGTHS[target]
    assert ssrows[0]==seq, f'IPKnot sequence does not match FASTA for {target}'
    assert len(ssrows)==2, f'Unexpected predicted SS line count in {ss}'
    dot=ssrows[1]
    assert len(dot)==len(seq), f'Dot-bracket length mismatch in {target}'
    mat=dbn_to_binary_bp(dot)
    return {'target_id':target,'sequence':seq,'dot_bracket':dot,'bp_map':mat,'source_fasta':fa,'source_ss':ss,
            'sha256_fasta':sha256(fa),'sha256_ipknot_ss':sha256(ss),'predicted_bp_pairs':int(mat.sum()/2)}

inputs={t:get_inputs(t) for t in TARGETS}
for t in TARGETS:
    x=inputs[t];print(f'{t:9s} length={len(x["sequence"]):2d}, predicted IPKnot pairs={x["predicted_bp_pairs"]:2d}')
print('FROZEN INPUTS PASS: 10/10, no native structures or native BP maps loaded')


## 3 · Install pinned official RNAbpFlow environment (once per fresh Colab runtime)

In [ ]:
def run_checked(args,cwd=None,env=None):
    print('$',' '.join(map(str,args)),flush=True)
    subprocess.run([str(a) for a in args],cwd=cwd,env=env,check=True)

REPO=Path('/content/RNAbpFlow')
if not REPO.exists():run_checked(['git','clone','https://github.com/Bhattacharya-Lab/RNAbpFlow.git',str(REPO)])
run_checked(['git','fetch','origin',UPSTREAM_SHA],cwd=REPO)
run_checked(['git','checkout','--detach',UPSTREAM_SHA],cwd=REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==UPSTREAM_SHA
MAMBA=Path('/content/bin/micromamba')
if not MAMBA.exists():
    run_checked(['bash','-lc','cd /content && curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj'])
assert MAMBA.is_file()
MAMBA_ROOT='/content/micromamba_root'
ENV_NAME='RNAbpFlow'
installed=subprocess.check_output([str(MAMBA),'-r',MAMBA_ROOT,'env','list'],text=True)
if 'RNAbpFlow' not in installed:
    run_checked([MAMBA,'-r',MAMBA_ROOT,'create','-y','-n',ENV_NAME,'-f',REPO/'RNAbpFlow.yml'])
run_checked([MAMBA,'-r',MAMBA_ROOT,'install','-n',ENV_NAME,'-y','-c','conda-forge','mkl<2024.1','numpy<2'])
check=subprocess.run([str(MAMBA),'-r',MAMBA_ROOT,'run','-n',ENV_NAME,'python','-c',
    'import torch,numpy;print("PyTorch",torch.__version__,"NumPy",numpy.__version__,"CUDA",torch.cuda.is_available());assert torch.cuda.is_available()'],capture_output=True,text=True)
print(check.stdout,check.stderr)
assert check.returncode==0,'GPU environment failed, stop before inference'


## 4 · Download the verified RNA3DB checkpoint (official Zenodo archive)

In [ ]:
WEIGHT=REPO/'checkpoint/RNA3DB.ckpt'
if not WEIGHT.is_file():
    ARCHIVE=Path('/content/Checkpoints.tar.gz')
    if not ARCHIVE.exists():run_checked(['wget','-q','--show-progress','-O',ARCHIVE,
        'https://zenodo.org/records/19388910/files/Checkpoints.tar.gz?download=1'])
    extracted=Path('/content/rnabpflow_ckpt_extract')
    extracted.mkdir(exist_ok=True)
    run_checked(['tar','-xzf',ARCHIVE,'-C',extracted])
    hits=list(extracted.rglob('RNA3DB.ckpt'))
    assert len(hits)>=1,'RNA3DB checkpoint missing in official archive'
    WEIGHT.parent.mkdir(exist_ok=True)
    shutil.copy2(hits[0],WEIGHT)
assert WEIGHT.stat().st_size>100_000_000, 'Checkpoint is unexpectedly small'
print('Checkpoint ready:',WEIGHT,'bytes:',WEIGHT.stat().st_size,'SHA256:',sha256(WEIGHT))


## 5 · Run 1 frozen target initially; save each completed prediction directly to Google Drive

In [ ]:
import yaml

def completed(target):
    folder=RUN_ROOT/target
    marker=folder/'complete.json'
    if not marker.exists():return False
    rec=json.loads(marker.read_text())
    assert rec['target_id']==target and rec['status']=='PASS' and rec['n_samples']==SAMPLES_PER_TARGET
    files=rec['prediction_files']
    assert len(files)==SAMPLES_PER_TARGET
    for item in files:
        p=folder/item['filename']
        assert p.is_file() and sha256(p)==item['sha256'],f'Saved result integrity error: {p}'
    return True

if SMOKE_ONLY:
    todo=[] if completed('9g4p_B') else ['9g4p_B']
else:
    todo=[t for t in TARGETS if not completed(t)][:TARGETS_PER_SESSION]
print('To run:',todo,'Completed:',sum(completed(t) for t in TARGETS),'/10')
assert SAMPLES_PER_TARGET==1,'Fixed one-sample baseline: do not silently change without new run ID'
assert todo or all(completed(t) for t in TARGETS) or SMOKE_ONLY

for target in todo:
    x=inputs[target];dest=RUN_ROOT/target
    assert not dest.exists(),f'Partial Drive output exists, inspect before rerun: {dest}'
    local_inputs=REPO/'Inputs'/target
    assert not local_inputs.exists(), f'Unexpected local input target directory {local_inputs}'
    local_inputs.mkdir(parents=True)
    shutil.copy2(x['source_fasta'],local_inputs/f'{target}.fasta')
    np.save(local_inputs/'map1.npy',x['bp_map'],allow_pickle=False)
    assert np.array_equal(np.load(local_inputs/'map1.npy'),x['bp_map'])
    (REPO/'Inputs/list.txt').write_text(f'{target} {SAMPLES_PER_TARGET}\n')
    # Restrict to a single GPU and one sample; output format PDB.
    config_path=REPO/'configs/inference.yaml'
    cfg=yaml.safe_load(config_path.read_text())
    cfg['inference']['num_gpus']=1
    cfg['inference']['input_dir']='Inputs'
    cfg['inference']['output_dir']='Predictions'
    cfg['inference']['ckpt_path']='checkpoint/RNA3DB.ckpt'
    cfg['inference']['output_format']='pdb'
    cfg['inference']['samples']['samples_per_sequence']=SAMPLES_PER_TARGET
    config_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
    output=REPO/'Predictions'/target
    assert not output.exists(),f'Prediction directory already exists: {output}. Inspect before rerun.'
    logpath=Path('/content')/f'rnabpflow_{target}.log'
    readings=[];stop=threading.Event()
    def mon():
        while not stop.wait(2):
            try:
                mem=subprocess.check_output(['nvidia-smi','-i','0','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True,timeout=15)
                readings.append(int(mem.strip().splitlines()[0]))
            except Exception:pass
    thread=threading.Thread(target=mon,daemon=True);thread.start()
    print('START RNAbpFlow:',target,'length',len(x['sequence']),'predicted IPKnot pairs',x['predicted_bp_pairs'],flush=True)
    start=time.monotonic()
    try:
        with logpath.open('w') as out:
            p=subprocess.run([str(MAMBA),'-r',MAMBA_ROOT,'run','-n',ENV_NAME,'python','inference.py'],cwd=REPO,stdout=out,stderr=subprocess.STDOUT)
    finally:
        stop.set();thread.join(timeout=5)
    elapsed=round(time.monotonic()-start,3)
    if p.returncode!=0:
        print(logpath.read_text(errors='replace')[-8000:])
        raise RuntimeError(f'RNAbpFlow inference failed: {target}; exit code {p.returncode}')
    structures=sorted(output.rglob('*.pdb'))
    assert len(structures)==SAMPLES_PER_TARGET, f'Expected {SAMPLES_PER_TARGET} PDB(s), got {len(structures)}: {structures}'
    # Single sample selection avoids native-guided selection bias.
    for s in structures:
        residues={(line[21:22],line[22:26],line[26:27]) for line in s.read_text(errors='replace').splitlines() if line.startswith('ATOM  ')}
        assert len(residues)==len(x['sequence']),f'Unexpected residue count: {s}, {len(residues)}'
    local_stage=Path('/content')/'rnabpflow_stage'/target
    assert not local_stage.exists()
    local_stage.mkdir(parents=True)
    shutil.copy2(x['source_fasta'],local_stage/f'{target}.fasta')
    shutil.copy2(x['source_ss'],local_stage/f'{target}.single.ipknot.ss')
    shutil.copy2(local_inputs/'map1.npy',local_stage/'map1.npy')
    shutil.copy2(logpath,local_stage/'inference.log')
    info=[]
    for i,s in enumerate(structures):
        name=f'{target}_sample_{i}.pdb'
        shutil.copy2(s,local_stage/name)
        info.append({'filename':name,'sha256':sha256(local_stage/name),'residues':len(x['sequence'])})
    record={'status':'PASS','model':'RNAbpFlow','target_id':target,'n_samples':SAMPLES_PER_TARGET,
        'input_length':len(x['sequence']),'input_mode':'fasta_predicted_ipknot_single_binary_bp_map',
        'bp_source':'frozen NuFold pilot10 *.single.ipknot.ss (predicted, not native)',
        'predicted_bp_pairs':x['predicted_bp_pairs'],
        'model_commit':UPSTREAM_SHA,'model_checkpoint':'RNA3DB.ckpt','checkpoint_sha256':sha256(WEIGHT),
        'fastasha256':x['sha256_fasta'],'ipknot_ss_sha256':x['sha256_ipknot_ss'],
        'map1_sha256':sha256(local_stage/'map1.npy'),
        'runtime_seconds':elapsed,'peak_gpu_memory_mib_observed':max(readings) if readings else None,
        'prediction_files':info,'native_used_for_inference':False,'native_bp_used_for_inference':False,
        'exit_code':p.returncode,'created_utc':datetime.now(timezone.utc).isoformat()}
    (local_stage/'complete.json').write_text(json.dumps(record,indent=2)+'\n')
    # Drive copy: use a unique target directory and final marker last.
    dest.mkdir(parents=True,exist_ok=False)
    for s in local_stage.iterdir():
        if s.name!='complete.json':shutil.copy2(s,dest/s.name)
    for it in info:
        assert sha256(dest/it['filename'])==it['sha256']
    shutil.copy2(local_stage/'complete.json',dest/'complete.json')
    assert completed(target)
    print('SAVED AND VERIFIED ON DRIVE:',target,'seconds:',elapsed,'PDB:',info[0]['filename'],flush=True)

print('RUN FINISHED. Verified targets:',sum(completed(t) for t in TARGETS),'/10')


## 6 · Read-only verification and next session

In [ ]:
rows=[]
for t in TARGETS:
    try:status='PASS' if completed(t) else 'PENDING'
    except Exception as ex:status='INTEGRITY ERROR: '+str(ex)
    rows.append({'target_id':t,'status':status})
for row in rows:print(row['target_id'],row['status'])
print('Drive run folder:',RUN_ROOT)
print('If smoke passed, set SMOKE_ONLY=False in Cell 1 and rerun the scope + inference cells (reuse environment in same runtime).')
print('No GitHub write, no native structures read, no evaluation performed in this notebook.')
